<a href="https://colab.research.google.com/github/knatarajan7/GenBus885_Assignment7_Natarajan_K/blob/main/GenBus885_Final_Project_Natarajan_K.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Rush Sportwear and Footwear
## Trend and Insight analysis
Analyze sales data for trends and insights that will help company leadership understand the market and identify opportunities for growth.


In [ ]:
# Import the appropriate Python libraries.
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Load the Products CSV file into a dataframe.
df_products = pd.read_csv('https://raw.githubusercontent.com/knatarajan7/rush_case_study/refs/heads/main/TABLE_PRODUCTS_885.csv',delimiter='|')
df_products.head()

In [ ]:
#Check the dataframe information
df_products.info()

In [ ]:
#check if there are any duplicates for Product_ID since it will be used to join with other tables
df_products['PRODUCT_ID'].duplicated().sum()

In [ ]:
# Load the Retailer CSV file into a dataframe.
df_retailer = pd.read_csv('https://raw.githubusercontent.com/knatarajan7/rush_case_study/refs/heads/main/TABLE_RETAILER_885.csv')
df_retailer.head()

In [ ]:
#Check the dataframe information
df_retailer.info()

In [ ]:
#Check for duplicates of Retailer ID since this will be used to join with other tables
df_retailer['RETAILER_ID'].duplicated().sum()

In [ ]:
#Find the Duplicates
df_retailer['RETAILER_ID'].value_counts().sort_values(ascending=False)

In [ ]:
#Need to drop one row each for W00SFLOR,W00SARLI,W00STEHO,S00NNENE because there is no good indicator on which one is the right one.
df_retailer[df_retailer['RETAILER_ID'].isin(['W00SFLOR', 'W00STEHO','W00SARLI','S00NNENE'])]

In [ ]:
#Drop rows with Index: 64, 81, 83, 84 otherwise we will have duplication of rows when joining. Verify with data experts later.
df_retailer=df_retailer.drop(index=64,errors='ignore')
df_retailer=df_retailer.drop(index=81,errors='ignore')
df_retailer=df_retailer.drop(index=83,errors='ignore')
df_retailer=df_retailer.drop(index=84,errors='ignore')
df_retailer['RETAILER_ID'].duplicated().sum()

In [ ]:
df_retailer['REGION'].value_counts()

In [ ]:
df_retailer['STATE'].value_counts()

In [ ]:
df_retailer['CITY'].value_counts()

In [ ]:
# Load the Sales CSV file into a dataframe and view a few rows
df_sales = pd.read_csv('https://raw.githubusercontent.com/knatarajan7/rush_case_study/refs/heads/main/TABLE_SALES_885.csv')
df_sales.head()

In [ ]:
#Check the dataframe information
df_sales.info()

In [ ]:
#Price_per_unit is NaN for two rows. this will need to be taken care of.
#view a few rows
df_sales.head()

In [ ]:
#drop Day column since it will not be useful for analysis
df_sales = df_sales.drop(columns=['DAY'])
df_sales.describe()

In [ ]:
df_sales['PRICE_PER_UNIT'].sort_values(ascending=False)

In [ ]:
#find the row with the anomalous Price_per_unit
df_sales[df_sales['PRICE_PER_UNIT'] == 99999.0]

In [ ]:
#find the median value for Price_per_unit
df_sales.loc[df_sales['PRICE_PER_UNIT'] != 99999.0, 'PRICE_PER_UNIT'].median()

In [ ]:
#replace the value 99999.0 and the NaN with median
df_sales['PRICE_PER_UNIT'] = df_sales['PRICE_PER_UNIT'].replace(99999.0, 45.0)
df_sales['PRICE_PER_UNIT'] = df_sales['PRICE_PER_UNIT'].fillna(45.0)
#replace the price per unit as integer as no fractional values were found
df_sales['PRICE_PER_UNIT'] = df_sales['PRICE_PER_UNIT'].astype(int)
df_sales.describe()

In [ ]:
df_sales.info()

In [ ]:
#find any unusual values for UNITS_SOLD
df_sales['UNITS_SOLD'].sort_values()


In [ ]:
#delete rows where UNITS_SOLD is '***' - now the row count is 9648-2 = 9646
df_sales = df_sales[df_sales['UNITS_SOLD'] != '***'].copy()
df_sales.info()

In [ ]:
#reformat the Units_sold to int
df_sales['UNITS_SOLD'] = df_sales['UNITS_SOLD'].astype(int)

In [ ]:
#check the operating_margin for any unusual values
df_sales['OPERATING_MARGIN'].sort_values().value_counts()

In [ ]:
#check the sales_method column for any unusual values
df_sales['SALES_METHOD'].sort_values().value_counts()

In [ ]:
#Fix typo 'Ootlet' with 'Outlet'
df_sales['SALES_METHOD'] = df_sales['SALES_METHOD'].replace('Ootlet', 'Outlet')
df_sales['SALES_METHOD'].sort_values().value_counts()

In [ ]:
#Check unique values of Retailer_ID in df_sales
df_sales['RETAILER_ID'].unique()

In [ ]:
#Check the number of rows with the unknown Retailer_ID
df_sales[df_sales['RETAILER_ID'] == '999999999']

In [ ]:
#remove the row with the incorrect Retailer_ID and check again - row count is 9646-1 = 9645
df_sales = df_sales[df_sales['RETAILER_ID'] != '999999999'].copy()
df_sales[df_sales['RETAILER_ID'] == '999999999']

In [ ]:
#check all values for Product_id to checck if they are valid
df_sales['PRODUCT_ID'].unique()

In [ ]:
#convert the invoice dates to datetime and check if any are invalid or missing
df_sales['INVOICE_DATE'] = pd.to_datetime(df_sales['INVOICE_DATE'],format='%m/%d/%Y',errors='coerce')
df_sales['INVOICE_DATE'].isna().sum()

In [ ]:
#add a column for year and month
#add a column to calculate totla revenue for each row
df_sales['YEAR_MONTH'] = df_sales['INVOICE_DATE'].dt.strftime('%Y-%m')
df_sales['TOTAL_REVENUE'] = df_sales['PRICE_PER_UNIT'] * df_sales['UNITS_SOLD']
df_sales.head()

In [ ]:
#right join to the description tables and check if there are any duplicates because of the join - the row count should be 9645 after join
df_sales_retailer = pd.merge(df_sales, df_retailer, on='RETAILER_ID', how='right')
df_sales_retailer_product = pd.merge(df_sales_retailer, df_products, on='PRODUCT_ID', how='right')
df_sales_retailer_product.info()

In [ ]:
#What product category (product) had the highest sales (in dollars) in 2021? How much did it sell?
df_test = df_sales_retailer_product[df_sales_retailer_product['YEAR'] == 2021].groupby(['YEAR','PRODUCT_ID','PRODUCT_NAME'])['TOTAL_REVENUE'].sum().reset_index()
df_test
#Men's Street Footwear - 22654920

In [ ]:
#Create a column to identify Men's products and Women's products

df_sales_retailer_product['GENDER'] = np.where(
    df_sales_retailer_product['PRODUCT_ID'].isin([20, 30, 40]), 'Men',
    'Women')

In [ ]:
#What state had the highest sales (in dollars) of women's products in 2021? How much was it?
df_sales_retailer_product[(df_sales_retailer_product['GENDER'] == 'Women') & (df_sales_retailer_product['YEAR'] == 2021)].groupby(['YEAR','STATE','GENDER'])['TOTAL_REVENUE'].sum().sort_values(ascending=False)

In [ ]:
#What state had the highest sales (in dollars) of men's products in 2021? How much was it?
df_sales_retailer_product[(df_sales_retailer_product['GENDER'] == 'Men') & (df_sales_retailer_product['YEAR'] == 2021)].groupby(['YEAR','STATE','GENDER'])['TOTAL_REVENUE'].sum().sort_values(ascending=False)

In [ ]:
#What retailer purchased the most units in 2021?
df_sales_retailer_product[(df_sales_retailer_product['YEAR'] == 2021)].groupby(['RETAILER'])['TOTAL_REVENUE'].sum().sort_values(ascending=False)

In [ ]:
#In 2020?
df_sales_retailer_product[(df_sales_retailer_product['YEAR'] == 2020)].groupby(['RETAILER'])['TOTAL_REVENUE'].sum().sort_values(ascending=False)

In [ ]:
#Sales by Month and year
df_sales_monthly = df_sales_retailer_product.groupby(['YEAR_MONTH'])['TOTAL_REVENUE'].sum().reset_index()
sns.lineplot(data=df_sales_monthly, x='YEAR_MONTH', y='TOTAL_REVENUE')
plt.xticks(rotation=90)
plt.show()

In [ ]:
#Sales by Month, Year, Sales Method
df_sales_monthly = df_sales_retailer_product.groupby(['YEAR_MONTH', 'SALES_METHOD'])['TOTAL_REVENUE'].sum().reset_index()
sns.lineplot(data=df_sales_monthly, x='YEAR_MONTH', y='TOTAL_REVENUE', hue='SALES_METHOD')
plt.xticks(rotation=90)
plt.show()

In [ ]:
#Sales by Month, Year, Region
df_sales_monthly = df_sales_retailer_product.groupby(['YEAR_MONTH', 'REGION'])['TOTAL_REVENUE'].sum().reset_index()
sns.lineplot(data=df_sales_monthly, x='YEAR_MONTH', y='TOTAL_REVENUE', hue='REGION')
plt.xticks(rotation=90)
plt.show()

In [ ]:
#Sales by Month, Year, Product
df_sales_monthly = df_sales_retailer_product.groupby(['YEAR_MONTH', 'PRODUCT_NAME'])['TOTAL_REVENUE'].sum().reset_index()
sns.lineplot(data=df_sales_monthly, x='YEAR_MONTH', y='TOTAL_REVENUE', hue='PRODUCT_NAME')
plt.xticks(rotation=90)
plt.show()

In [ ]:
#Sales by Month, Year, Retailer
df_sales_monthly = df_sales_retailer_product.groupby(['YEAR_MONTH', 'RETAILER'])['TOTAL_REVENUE'].sum().reset_index()
sns.lineplot(data=df_sales_monthly, x='YEAR_MONTH', y='TOTAL_REVENUE', hue='RETAILER')
plt.xticks(rotation=90)
plt.show()